# LangChain Model Switcher - AI Math Assistant

This notebook demonstrates how to build an AI math assistant that can work with multiple LLM providers:
- IBM Watson (Granite models)
- Anthropic Claude
- Ollama (local models)

Switch between models by setting the `MODEL_PROVIDER` environment variable!

## Setup and Configuration

In [ ]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Add src to path for imports
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

print(f"Project root: {project_root}")
print(f"Current working directory: {Path.cwd()}")

In [ ]:
# Import our model switcher
from utils.model_factory import ModelFactory, get_model
from config.settings import get_current_provider, get_model_config
from mcp.tools import get_math_tools

# Set the model provider - change this to switch models!
# Options: "watson", "claude", "ollama"
MODEL_PROVIDER = "claude"  # Change this to test different models
os.environ["MODEL_PROVIDER"] = MODEL_PROVIDER

print(f"Using model provider: {MODEL_PROVIDER}")
print(f"Available providers: {ModelFactory.list_available_providers()}")

## Model Initialization

The beauty of our model switcher is that this single line works with any provider!

In [ ]:
# Get the current model - this works with any provider!
model_adapter = get_model()
llm = model_adapter.get_langchain_model()

print(f"Model adapter: {model_adapter}")
print(f"Model name: {model_adapter.get_model_name()}")
print(f"Supports tool calling: {model_adapter.supports_tool_calling()}")
print(f"LangChain model: {type(llm).__name__}")

## Test Basic Model Functionality

In [ ]:
# Test the model with a simple query
response = llm.invoke("What is tool calling in LangChain? Answer in 2 sentences.")
print("Model Response:")
print(response.content)

## Load Mathematical Tools

These tools work with any model provider through our unified interface.

In [ ]:
# Get all mathematical tools
tools = get_math_tools()

print("Available tools:")
for tool in tools:
    print(f"- {tool.name}: {tool.description}")

## Create the Math Agent

This agent works seamlessly with any model provider!

In [ ]:
from langgraph.prebuilt import create_react_agent

# Create the agent with all tools
math_agent = create_react_agent(
    model=llm,  # This now works with any provider!
    tools=tools,
    prompt="You are a helpful mathematical assistant that can perform various operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

print(f"Math agent created with {len(tools)} tools")
print(f"Agent is using: {model_adapter.get_model_name()}")

## Test Mathematical Operations

In [ ]:
# Test addition
response = math_agent.invoke({
    "messages": [("human", "Add the numbers 25, 15, and 10")]
})

print("Addition Test:")
print(f"Query: Add the numbers 25, 15, and 10")
print(f"Answer: {response['messages'][-1].content}")

In [ ]:
# Test multiplication
response = math_agent.invoke({
    "messages": [("human", "Multiply 6, 7, and 2")]
})

print("Multiplication Test:")
print(f"Query: Multiply 6, 7, and 2")
print(f"Answer: {response['messages'][-1].content}")

In [ ]:
# Test division
response = math_agent.invoke({
    "messages": [("human", "Divide 144 by 12 and then by 3")]
})

print("Division Test:")
print(f"Query: Divide 144 by 12 and then by 3")
print(f"Answer: {response['messages'][-1].content}")

In [ ]:
# Test subtraction
response = math_agent.invoke({
    "messages": [("human", "Subtract 30 and 15 from 100")]
})

print("Subtraction Test:")
print(f"Query: Subtract 30 and 15 from 100")
print(f"Answer: {response['messages'][-1].content}")

## Test Complex Multi-Step Operations

In [ ]:
# Test complex operation
complex_query = "Calculate (25 + 15) multiplied by 3, then subtract 20"

response = math_agent.invoke({
    "messages": [("human", complex_query)]
})

print("Complex Operation Test:")
print(f"Query: {complex_query}")
print(f"Answer: {response['messages'][-1].content}")

## Test Wikipedia Integration

In [ ]:
# Test Wikipedia search with mathematical calculation
wiki_math_query = "What is the population of Canada? Then multiply it by 0.25"

response = math_agent.invoke({
    "messages": [("human", wiki_math_query)]
})

print("Wikipedia + Math Test:")
print(f"Query: {wiki_math_query}")
print(f"Answer: {response['messages'][-1].content}")

## Model Comparison Demo

Let's test the same query with different models to see how they perform!

In [ ]:
def test_model_with_query(provider: str, query: str):
    """Test a specific model provider with a query."""
    try:
        # Set the provider
        os.environ["MODEL_PROVIDER"] = provider
        
        # Get the model
        adapter = get_model(provider)
        model = adapter.get_langchain_model()
        
        # Create agent
        agent = create_react_agent(
            model=model,
            tools=tools,
            prompt="You are a helpful mathematical assistant."
        )
        
        # Test the query
        response = agent.invoke({"messages": [("human", query)]})
        
        return {
            "provider": provider,
            "model_name": adapter.get_model_name(),
            "success": True,
            "response": response['messages'][-1].content
        }
    except Exception as e:
        return {
            "provider": provider,
            "model_name": "N/A",
            "success": False,
            "error": str(e)
        }

# Test query
test_query = "Add 15 and 25, then multiply by 2"

print(f"Testing query: '{test_query}'\n")
print("=" * 80)

# Test each available provider
providers_to_test = ["claude", "watson", "ollama"]

for provider in providers_to_test:
    print(f"\nTesting {provider.upper()}:")
    print("-" * 40)
    
    result = test_model_with_query(provider, test_query)
    
    if result["success"]:
        print(f"✅ Model: {result['model_name']}")
        print(f"Response: {result['response']}")
    else:
        print(f"❌ Error with {provider}: {result['error']}")

# Reset to original provider
os.environ["MODEL_PROVIDER"] = MODEL_PROVIDER

## Configuration Information

In [ ]:
# Show current configuration
current_provider = get_current_provider()
current_config = get_model_config(current_provider)

print("Current Configuration:")
print(f"Provider: {current_provider}")
print(f"Model ID: {current_config.model_id}")
print(f"Adapter Class: {current_config.class_name}")
print(f"Temperature: {current_config.temperature}")
print(f"Max Tokens: {current_config.max_tokens}")

# Show all available providers
print(f"\nAll Available Providers: {ModelFactory.list_available_providers()}")

## Quick Provider Switch Demo

See how easy it is to switch between models!

In [ ]:
# Function to quickly switch and test
def quick_switch_test(new_provider: str):
    """Quickly switch provider and test with a simple calculation."""
    os.environ["MODEL_PROVIDER"] = new_provider
    
    try:
        adapter = get_model()
        print(f"✅ Switched to: {adapter.get_model_name()}")
        
        # Quick test
        llm_test = adapter.get_langchain_model()
        response = llm_test.invoke("What is 5 + 3?")
        print(f"Quick test response: {response.content}")
        
    except Exception as e:
        print(f"❌ Error switching to {new_provider}: {e}")

print("Testing provider switching:")
print("\n1. Testing Claude:")
quick_switch_test("claude")

print("\n2. Testing Watson:")
quick_switch_test("watson")

print("\n3. Testing Ollama:")
quick_switch_test("ollama")

# Reset to original
os.environ["MODEL_PROVIDER"] = MODEL_PROVIDER
print(f"\nReset to: {MODEL_PROVIDER}")

## Vector Database Applications

Your chunk store applications can now use any model! Here's a preview of how you'd integrate with a vector database:

In [ ]:
# Example of how to use with vector databases
# This demonstrates the concept - actual implementation would require vector store setup

print("Vector Database Integration Example:")
print("====================================")
print()
print("# Example code for vector database integration:")
print()
print("from langchain_community.vectorstores import Chroma")
print("from langchain.embeddings import OpenAIEmbeddings")
print("from langchain.text_splitter import RecursiveCharacterTextSplitter")
print("from utils.model_factory import get_model")
print()
print("# Get any model for your RAG system")
print("llm = get_model().get_langchain_model()")
print()
print("# Set up vector store (works with any model!)")
print("embeddings = OpenAIEmbeddings()")
print("vectorstore = Chroma(embedding_function=embeddings)")
print()
print("# Create retrieval chain")
print("from langchain.chains import RetrievalQA")
print("qa_chain = RetrievalQA.from_chain_type(")
print("    llm=llm,  # Any model works here!")
print("    chain_type='stuff',")
print("    retriever=vectorstore.as_retriever()")
print(")")
print()
print("# Switch models anytime:")
print("# os.environ['MODEL_PROVIDER'] = 'claude'  # or 'watson', 'ollama'")
print("# llm = get_model().get_langchain_model()")
print("# qa_chain.llm = llm  # Update the chain with new model")

current_model = get_model()
print(f"\nCurrently configured for: {current_model.get_model_name()}")
print(f"This model {'✅ supports' if current_model.supports_tool_calling() else '❌ does not support'} tool calling")

## Summary

🎉 **Congratulations!** You've successfully created a unified LangChain interface that works with multiple LLM providers:

### Key Benefits:
- **🔄 Easy Switching**: Change models with one environment variable
- **🧩 Unified Interface**: Same code works with Watson, Claude, and Ollama
- **🛠️ MCP Ready**: Tools are exposed via Model Context Protocol
- **📈 Scalable**: Easy to add new model providers
- **🏗️ Vector Database Ready**: Perfect foundation for RAG applications

### Next Steps:
1. **Vector Database Integration**: Build RAG systems with your chunk store
2. **Custom Tools**: Add domain-specific tools for your use case
3. **MCP Server**: Deploy the MCP server for external access
4. **Production Setup**: Configure API keys and deploy

### Model Switching Commands:
```bash
# Switch to Claude
export MODEL_PROVIDER=claude

# Switch to Watson
export MODEL_PROVIDER=watson

# Switch to Ollama
export MODEL_PROVIDER=ollama
```

Your chunk store applications will work seamlessly with any of these models! 🚀